# 01 — Data quality: what the two files actually contain

**Question this notebook answers:** can these two published files be merged into one
trustworthy table of 10-minute game states?

The short answer is yes, but only after removing about 9.5% of one of them. This notebook
shows how that was found, because the finding is the backbone of our story: *a dataset's
own label is not a measurement.*

Every number below is printed by the cell above it. Nothing here is typed by hand.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from lol import load, clean, features, viz

viz.set_style()
pd.set_option("display.width", 160)
load.check_raw_files()

,file,present,size_mb
0,high_diamond_ranked_10min.csv,True,1.45
1,Challenger_Ranked_Games_10minute.csv,True,4.10


## Part A — the two files side by side

Two authors, two vocabularies, two different column counts for the same game.

In [2]:
D = load.load_diamond()
C = load.load_challenger()

pd.DataFrame({
    "file":    ["Diamond–Master", "Challenger"],
    "rows":    [len(D), len(C)],
    "columns": [D.shape[1], C.shape[1]],
    "missing values":  [int(D.isna().sum().sum()), int(C.isna().sum().sum())],
    "duplicate gameId": [int(D.gameId.duplicated().sum()), int(C.gameId.duplicated().sum())],
    "blue win rate":   [round(D.blueWins.mean(), 4), round(C.blueWins.mean(), 4)],
})

,file,rows,columns,missing values,duplicate gameId,blue win rate
0,Diamond–Master,9879,40,0,0,0.4990
1,Challenger,26409,51,0,55,0.5011


Both are balanced and neither has a missing value. On that evidence alone we would have
merged them and moved on. The next cell is why we did not.

## Part B — the discovery

The Challenger file carries columns Diamond does not: towers by lane, first inhibitor,
**inhibitor destroyed**. An inhibitor sits behind three towers. It cannot fall by minute ten.

In [3]:
pd.DataFrame({
    "column": ["blueInhibitor", "blueFirstInhibitor", "blueTowerKills", "blueFirstTower"],
    "max":    [C.blueInhibitor.max(), C.blueFirstInhibitor.max(), C.blueTowerKills.max(), C.blueFirstTower.max()],
    "share non-zero": [f"{(C[c] != 0).mean():.2%}" for c in
                       ["blueInhibitor", "blueFirstInhibitor", "blueTowerKills", "blueFirstTower"]],
})

,column,max,share non-zero
0,blueInhibitor,1,0.62%
1,blueFirstInhibitor,1,0.62%
2,blueTowerKills,9,8.85%
3,blueFirstTower,1,7.11%


That prompted checking quantities with a hard physical ceiling. A team cannot be level 12
or hold 50,000 gold at ten minutes, in any division.

In [4]:
pd.DataFrame({
    "percentile": ["p50", "p90", "p95", "p99", "max"],
    "Challenger avg level": [round(C.blueAvgLevel.quantile(q), 2) for q in (.5, .9, .95, .99, 1.0)],
    "Challenger team gold": [round(C.blueTotalGolds.quantile(q)) for q in (.5, .9, .95, .99, 1.0)],
    "Diamond avg level":    [round(D.blueAvgLevel.quantile(q), 2) for q in (.5, .9, .95, .99, 1.0)],
    "Diamond team gold":    [round(D.blueTotalGold.quantile(q)) for q in (.5, .9, .95, .99, 1.0)],
})

,percentile,Challenger avg level,Challenger team gold,Diamond avg level,Diamond team gold
0,p50,7.0,16609,7.0,16398
1,p90,7.6,21101,7.2,18496
2,p95,10.8,33073,7.4,19190
3,p99,11.4,40145,7.6,20612
4,max,12.4,53161,8.0,23701


Level jumps from 7.60 at p90 to 10.80 at p95 and gold from 21,101 to 33,073 over the same
gap. A heavy tail fills in gradually; this does not. **There is no mass between the two
modes, so there are two populations in one file.** Diamond, by contrast, caps at 8.0.

In [5]:
fig = None
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7.4, 3.9))
ax.hist(C.blueAvgLevel, bins=np.arange(4, 13.1, 0.2), color=viz.SERIES_1, edgecolor="none")
ax.axvline(clean.MAX_AVG_LEVEL_AT_10MIN, color=viz.SERIES_2, lw=2, ls="--")
ax.set_title("The Challenger file holds two different populations")
ax.set_xlabel("Blue team average champion level, as published")
ax.set_ylabel("Games")
viz.thousands(ax)
plt.show()

<Figure size 814x429 with 1 Axes>

## Part C — the filter, and checking it is honest

Rows are removed where either team exceeds a physical ceiling, or an inhibitor is already
down. The thresholds sit *inside the empty gap* so the cut does not trim a real tail.

Dropping rows on variables correlated with the outcome can bias the target, so we measure
rather than assume.

In [6]:
bad = clean.flag_not_ten_minutes(C)
pd.DataFrame({
    "slice":        ["all Challenger rows", "removed (not 10-min)", "kept"],
    "rows":         [len(C), int(bad.sum()), int((~bad).sum())],
    "share":        ["100.00%", f"{bad.mean():.2%}", f"{(~bad).mean():.2%}"],
    "blue win rate": [round(C.blueWins.mean(), 4),
                      round(C.loc[bad, "blueWins"].mean(), 4),
                      round(C.loc[~bad, "blueWins"].mean(), 4)],
})

,slice,rows,share,blue win rate
0,all Challenger rows,26409,100.00%,0.5011
1,removed (not 10-min),2498,9.46%,0.5140
2,kept,23911,90.54%,0.4998


The target moves by 0.0013. The filter corrects a validity error without reshaping the
thing we are trying to predict.

### Two anomalies that turned out to be one

The dragon counts disagreed between files ({0,1} vs {0,1,2}) and Challenger had games with
1,318 wards placed. Both looked like separate problems needing separate rules. They were
symptoms of the same contamination.

In [7]:
Ck = C.loc[~bad]
pd.DataFrame({
    "measure":            ["dragon values", "max wards placed", "rows over 100 wards"],
    "before the filter":  [str(sorted(C.blueDragon.unique())), f"{C.blueWardPlaced.max():,}", f"{(C.blueWardPlaced > 100).sum():,}"],
    "after the filter":   [str(sorted(Ck.blueDragon.unique())), f"{Ck.blueWardPlaced.max():,}", f"{(Ck.blueWardPlaced > 100).sum():,}"],
})

,measure,before the filter,after the filter
0,dragon values,"[0, 1, 2]","[0, 1]"
1,max wards placed,"1,318",257
2,rows over 100 wards,132,59


The dragon mismatch disappears entirely — the concepts did match all along. The ward tail
shrinks but survives, in **both** files, inside otherwise-ordinary games:

In [8]:
hi = D.blueWardsPlaced > 100
pd.DataFrame({
    "Diamond games":  ["over 100 wards", "all others"],
    "rows":           [int(hi.sum()), int((~hi).sum())],
    "median gold":    [round(D.loc[hi, "blueTotalGold"].median()), round(D.loc[~hi, "blueTotalGold"].median())],
    "median level":   [D.loc[hi, "blueAvgLevel"].median(), D.loc[~hi, "blueAvgLevel"].median()],
    "median CS":      [D.loc[hi, "blueTotalMinionsKilled"].median(), D.loc[~hi, "blueTotalMinionsKilled"].median()],
})

,Diamond games,rows,median gold,median level,median CS
0,over 100 wards,108,16533,7.0,215.0
1,all others,9771,16398,7.0,218.0


Normal gold, normal level, normal CS. These are valid games with one broken column, so we
**cap** the ward columns rather than discard the games (decision D08).

## Part D — the merge

Thirteen shared concepts under different spellings. Challenger has no experience columns at
all, so those stay Diamond-only and are excluded from anything cross-tier.

In [9]:
pd.DataFrame(
    [{"canonical": k, "Diamond": f"blue{v[0]}", "Challenger": f"blue{v[1]}"} for k, v in clean.COLUMN_MAP.items()]
)

,canonical,Diamond,Challenger
0,wards_placed,blueWardsPlaced,blueWardPlaced
1,wards_destroyed,blueWardsDestroyed,blueWardKills
2,first_blood,blueFirstBlood,blueFirstBlood
3,kills,blueKills,blueKill
4,deaths,blueDeaths,blueDeath
5,assists,blueAssists,blueAssist
6,dragons,blueDragons,blueDragon
7,heralds,blueHeralds,blueRiftHeralds
8,towers_destroyed,blueTowersDestroyed,blueTowerKills
9,total_gold,blueTotalGold,blueTotalGolds


In [10]:
games, report = clean.build_games(verbose=False)
pd.Series(report).to_frame("value")

,value
diamond_rows_in,9879.0000
challenger_rows_in,26409.0000
challenger_not_10min,2498.0000
challenger_duplicate_ids,35.0000
red_wins_is_mirror,1.0000
diamond_duplicate_ids,0.0000
cross_tier_id_overlap,0.0000
ward_cells_capped,2935.0000
rows_out,33755.0000
blue_win_rate,0.4995


## Part E — 38 columns, 14 features

The headline. Rather than asserting which columns duplicate each other, we fit a line
between every pair and keep only those whose residual is exactly zero, then take the
measured rank of the whole block.

In [11]:
raw_cols = [c for c in D.columns if c not in ("gameId", "blueWins")]
dependent = features.audit_redundancy(D, raw_cols)
dependent[["relationship"]]

,relationship
0,redFirstBlood = -1·blueFirstBlood + 1
1,redDeaths = 1·blueKills + 1.30616e-15
2,redKills = 1·blueDeaths + 2.86808e-16
3,blueGoldPerMin = 0.1·blueTotalGold + -1.19985e-13
4,blueCSPerMin = 0.1·blueTotalMinionsKilled + -2...
5,redGoldDiff = -1·blueGoldDiff + -1.03213e-13
6,redExperienceDiff = -1·blueExperienceDiff + 1....
7,redGoldPerMin = 0.1·redTotalGold + 2.5446e-13
8,redCSPerMin = 0.1·redTotalMinionsKilled + 2.19...


Nine exact identities in the published file. The rank of the full block is lower still,
because some dependencies involve three columns rather than two
(`blueGoldDiff = blueTotalGold − redTotalGold`).

In [12]:
F = features.build_features(games)
fcols = features.feature_columns(F)

collapse = pd.DataFrame({
    "stage": ["published team columns", "carry independent information",
              "after the difference collapse", "modelled (one exact dependency dropped)"],
    "columns": [len(raw_cols),
                len(features.independent_set(D, raw_cols)),
                len(features.feature_columns(F, include_dependent=True)),
                len(fcols)],
})
collapse

,stage,columns
0,published team columns,38
1,carry independent information,25
2,after the difference collapse,15
3,modelled (one exact dependency dropped),14


The last step drops `deaths_diff`, because `blue_deaths` equals `red_kills` exactly, which
makes it identically `−kills_diff`. The audit found that; nobody judged it uninteresting.

In [13]:
remaining = features.audit_redundancy(F, fcols)
independent = features.independent_set(F, fcols)
print(f"modelling features:            {len(fcols)}")
print(f"exact dependencies remaining:  {len(remaining)}")
print(f"measured independent subset:   {len(independent)}")
assert len(remaining) == 0 and len(independent) == len(fcols)
print("\n" + ", ".join(fcols))

modelling features:            14
exact dependencies remaining:  0
measured independent subset:   14

wards_placed_diff, wards_destroyed_diff, first_blood_diff, kills_diff, assists_diff, dragons_diff, heralds_diff, towers_destroyed_diff, total_gold_diff, avg_level_diff, total_minions_diff, total_jungle_minions_diff, total_gold_both, total_kills_both


## Part F — the table handed to the rest of the team

In [14]:
out = load.processed_dir() / "games.parquet"
F.to_parquet(out, index=False)

pd.DataFrame({
    "property": ["rows", "Diamond games", "Challenger games", "modelling features",
                 "missing values in features", "blue win rate"],
    "value":    [f"{len(F):,}", f"{(F.tier == 'diamond').sum():,}", f"{(F.tier == 'challenger').sum():,}",
                 len(fcols), int(F[fcols].isna().sum().sum()), round(F.blue_wins.mean(), 4)],
})

,property,value
0,rows,"33,755"
1,Diamond games,"9,879"
2,Challenger games,"23,876"
3,modelling features,14
4,missing values in features,0
5,blue win rate,0.4995


## What this notebook established

1. **The Diamond file is clean** — no missing values, no duplicate ids, every quantity
   inside a plausible 10-minute range. It serves as the reference for what "10 minutes"
   looks like.
2. **The Challenger file is not purely a 10-minute snapshot.** About 9.5% of its rows record
   a later game state and were removed. Those rows carried post-10-minute information
   (towers, inhibitors, doubled gold) that would have leaked into Q1 and inflated its scores.
3. **Two further anomalies had the same single cause**, and needed no rules of their own.
4. **The residual ward tail is a genuine defect** in otherwise-valid games, so it is capped.
5. **38 published columns carry 14 features' worth of information.** Measured, not argued.

`tier` is kept in the table for grouping, and is deliberately **not** a modelling feature:
a model allowed to see it could learn which file a row came from instead of what the early
game says, which would make the Q4 transfer result meaningless.